# Merge electricity and weather data

In [1]:
import sys
import os
# Get the parent directory
parent_folder = os.path.dirname(os.getcwd())
# add the parent directory to the Python path so that the scripts can be imported
sys.path.append(parent_folder)

In [2]:
# Import libraries
from modules.utils import read_column_list_from_config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pvlib

In [3]:
f_name_elec = '../data/ausgrid/ausgrid_3_years.csv'
fname_ausgrid_meteo = "../data/ausgrid/ausgrid_cluster5_meteo.csv"
fname_ausgrid_meteo_location2 = "../data/ausgrid/ausgrid_meteo_location2_2010_2012.csv"
fname_ausgrid_meteo_location3_Jan = "../data/ausgrid/ausgrid_meteo_location3_2011_Jan.csv"
df_elec = pd.read_csv(f_name_elec)

df_meteo = pd.read_csv(
    fname_ausgrid_meteo,
    skipinitialspace=True,  # ignore spaces after comma separator
    usecols=[
        "period_end",
        "ghi",
        "azimuth",
        "zenith",
        "air_temp",
        "cloud_opacity",
        "wind_direction_10m",
        "wind_speed_10m",
        "precipitation_rate",
        "weather_type",
    ],
)
df_meteo_location2 = pd.read_csv(
    fname_ausgrid_meteo_location2,
    usecols=[
        "period_end",
        "ghi",
        "azimuth",
        "zenith",
        "air_temp",
        "cloud_opacity",
        "wind_direction_10m",
        "wind_speed_10m",
        "precipitation_rate",
        "weather_type",
    ],
)
df_meteo_location3_Jan = pd.read_csv(
    fname_ausgrid_meteo_location3_Jan,
    usecols=[
        "period_end",
        "ghi",
        "azimuth",
        "zenith",
        "air_temp",
        "cloud_opacity",
        "wind_direction_10m",
        "wind_speed_10m",
        "precipitation_rate",
        "weather_type",
    ],
)

In [4]:
df_elec.head()

,datetime,HoD,dow,doy,month,year,4_pv,4_con,4_net,6_pv,...,282_net,288_pv,288_con,288_net,291_pv,291_con,291_net,298_pv,298_con,298_net
0,2010-07-01 00:00:00,0,3,182,7,2010,0.0,150.0,150.0,0.0,...,188.0,0.0,82.0,82.0,0.0,796.0,796.0,0.0,192.0,192.0
1,2010-07-01 00:30:00,0,3,182,7,2010,0.0,172.0,172.0,0.0,...,176.0,0.0,26.0,26.0,0.0,826.0,826.0,0.0,220.0,220.0
2,2010-07-01 01:00:00,1,3,182,7,2010,0.0,170.0,170.0,0.0,...,176.0,0.0,80.0,80.0,0.0,552.0,552.0,0.0,160.0,160.0
3,2010-07-01 01:30:00,1,3,182,7,2010,0.0,168.0,168.0,0.0,...,150.0,0.0,28.0,28.0,0.0,614.0,614.0,0.0,210.0,210.0
4,2010-07-01 02:00:00,2,3,182,7,2010,0.0,168.0,168.0,0.0,...,176.0,0.0,78.0,78.0,0.0,528.0,528.0,0.0,178.0,178.0


In [5]:
df_elec.tail()

,datetime,HoD,dow,doy,month,year,4_pv,4_con,4_net,6_pv,...,282_net,288_pv,288_con,288_net,291_pv,291_con,291_net,298_pv,298_con,298_net
52603,2013-06-30 21:30:00,21,6,181,6,2013,0.0,268.0,268.0,0.0,...,1912.0,12.0,1628.0,1616.0,0.0,716.0,716.0,0.0,534.0,534.0
52604,2013-06-30 22:00:00,22,6,181,6,2013,0.0,274.0,274.0,0.0,...,1976.0,0.0,1608.0,1608.0,0.0,730.0,730.0,0.0,546.0,546.0
52605,2013-06-30 22:30:00,22,6,181,6,2013,0.0,280.0,280.0,0.0,...,1712.0,0.0,470.0,470.0,0.0,1144.0,1144.0,0.0,494.0,494.0
52606,2013-06-30 23:00:00,23,6,181,6,2013,0.0,224.0,224.0,0.0,...,1450.0,0.0,206.0,206.0,0.0,956.0,956.0,0.0,298.0,298.0
52607,2013-06-30 23:30:00,23,6,181,6,2013,0.0,214.0,214.0,0.0,...,1176.0,0.0,1088.0,1088.0,0.0,1242.0,1242.0,0.0,276.0,276.0


In [6]:
df_meteo_location3_Jan.head()

,air_temp,azimuth,cloud_opacity,ghi,precipitation_rate,wind_direction_10m,wind_speed_10m,zenith,weather_type,period_end
0,21,-175,0.0,0,0.0,10,4.0,123,CLEAR,2011-01-01T00:30:00+10:00
1,21,-167,0.0,0,0.0,8,4.0,122,CLEAR,2011-01-01T01:00:00+10:00
2,20,-160,0.0,0,0.0,6,3.9,120,CLEAR,2011-01-01T01:30:00+10:00
3,20,-152,0.0,0,0.0,4,3.8,117,CLEAR,2011-01-01T02:00:00+10:00
4,20,-145,0.0,0,0.0,2,3.6,114,CLEAR,2011-01-01T02:30:00+10:00


## Add datetime column to ausgrid_meteo

In [7]:
# Convert period_end to datetime column
df_meteo['datetime'] = pd.to_datetime(
    df_meteo['period_end']).dt.tz_localize(None)
df_meteo_location2['datetime'] = pd.to_datetime(
    df_meteo_location2['period_end']).dt.tz_convert('Etc/GMT-10').dt.tz_localize(None)
df_meteo_location3_Jan['datetime'] = pd.to_datetime(
    df_meteo_location3_Jan['period_end']).dt.tz_localize(None)

In [8]:
df_meteo.head()

,air_temp,azimuth,cloud_opacity,ghi,precipitation_rate,wind_direction_10m,wind_speed_10m,zenith,weather_type,period_end,datetime
0,6,-161,0.0,0,0.0,264,3.1,169,CLEAR,2010-06-30T00:30:00+10:00,2010-06-30 00:30:00
1,6,-134,0.0,0,0.0,268,3.1,165,CLEAR,2010-06-30T01:00:00+10:00,2010-06-30 01:00:00
2,6,-118,0.0,0,0.0,270,3.1,160,CLEAR,2010-06-30T01:30:00+10:00,2010-06-30 01:30:00
3,6,-108,0.0,0,0.0,271,3.2,154,CLEAR,2010-06-30T02:00:00+10:00,2010-06-30 02:00:00
4,5,-101,0.0,0,0.0,270,3.2,148,CLEAR,2010-06-30T02:30:00+10:00,2010-06-30 02:30:00


In [9]:
df_meteo_location2.head()

,air_temp,azimuth,cloud_opacity,ghi,precipitation_rate,wind_direction_10m,wind_speed_10m,zenith,weather_type,period_end,datetime
0,11,-27,0.0,451,0.0,285,3.3,62,SUNNY,2010-07-01T00:30:00+00:00,2010-07-01 10:30:00
1,12,-20,0.0,494,0.0,281,3.3,60,SUNNY,2010-07-01T01:00:00+00:00,2010-07-01 11:00:00
2,13,-12,1.5,515,0.0,275,3.3,58,MOSTLY SUNNY,2010-07-01T01:30:00+00:00,2010-07-01 11:30:00
3,13,-4,5.3,508,0.0,268,3.2,57,MOSTLY SUNNY,2010-07-01T02:00:00+00:00,2010-07-01 12:00:00
4,14,4,1.8,527,0.0,262,3.1,57,MOSTLY SUNNY,2010-07-01T02:30:00+00:00,2010-07-01 12:30:00


In [10]:
df_meteo['cloud_opacity'].describe()

count    52656.000000
mean        21.766790
std         30.161625
min          0.000000
25%          0.000000
50%          0.000000
75%         47.900000
max         97.000000
Name: cloud_opacity, dtype: float64

In [11]:
df_meteo_location3_Jan.head()

,air_temp,azimuth,cloud_opacity,ghi,precipitation_rate,wind_direction_10m,wind_speed_10m,zenith,weather_type,period_end,datetime
0,21,-175,0.0,0,0.0,10,4.0,123,CLEAR,2011-01-01T00:30:00+10:00,2011-01-01 00:30:00
1,21,-167,0.0,0,0.0,8,4.0,122,CLEAR,2011-01-01T01:00:00+10:00,2011-01-01 01:00:00
2,20,-160,0.0,0,0.0,6,3.9,120,CLEAR,2011-01-01T01:30:00+10:00,2011-01-01 01:30:00
3,20,-152,0.0,0,0.0,4,3.8,117,CLEAR,2011-01-01T02:00:00+10:00,2011-01-01 02:00:00
4,20,-145,0.0,0,0.0,2,3.6,114,CLEAR,2011-01-01T02:30:00+10:00,2011-01-01 02:30:00


In [12]:
# Read weather columns from config file
weather_columns = read_column_list_from_config('../config/weather_columns.txt')

In [13]:
columns_to_keep = ['datetime'] + weather_columns
df_meteo = df_meteo[columns_to_keep]

In [14]:
df_meteo_location2 = df_meteo_location2[columns_to_keep]
df_meteo_location2.head()

,datetime,air_temp,precipitation_rate,wind_speed_10m,ghi,azimuth,zenith,cloud_opacity
0,2010-07-01 10:30:00,11,0.0,3.3,451,-27,62,0.0
1,2010-07-01 11:00:00,12,0.0,3.3,494,-20,60,0.0
2,2010-07-01 11:30:00,13,0.0,3.3,515,-12,58,1.5
3,2010-07-01 12:00:00,13,0.0,3.2,508,-4,57,5.3
4,2010-07-01 12:30:00,14,0.0,3.1,527,4,57,1.8


In [15]:
# add _loc2 to the columns in df_meteo_location2, besides the datetime column
df_meteo_location2.columns = [
    f"{col}_loc2" if col != 'datetime' else col
    for col in df_meteo_location2.columns
]
df_meteo_location2.head()

,datetime,air_temp_loc2,precipitation_rate_loc2,wind_speed_10m_loc2,ghi_loc2,azimuth_loc2,zenith_loc2,cloud_opacity_loc2
0,2010-07-01 10:30:00,11,0.0,3.3,451,-27,62,0.0
1,2010-07-01 11:00:00,12,0.0,3.3,494,-20,60,0.0
2,2010-07-01 11:30:00,13,0.0,3.3,515,-12,58,1.5
3,2010-07-01 12:00:00,13,0.0,3.2,508,-4,57,5.3
4,2010-07-01 12:30:00,14,0.0,3.1,527,4,57,1.8


In [16]:
df_meteo_location3_Jan = df_meteo_location3_Jan[columns_to_keep]
df_meteo_location3_Jan.columns = [
    f"{col}_loc3" if col != 'datetime' else col
    for col in df_meteo_location3_Jan.columns
]

## Merge the electricity and weather data

In [17]:
# Ensure both datetime columns are the same type before merging
df_elec['datetime'] = pd.to_datetime(df_elec['datetime'])
df_meteo['datetime'] = pd.to_datetime(df_meteo['datetime'])
df_meteo_location2['datetime'] = pd.to_datetime(df_meteo_location2['datetime'])
df_meteo_location3_Jan['datetime'] = pd.to_datetime(
    df_meteo_location3_Jan['datetime'])
# Merge electricity and weather data on datetime column
df_merged = pd.merge(df_elec, df_meteo, on='datetime', how='left')
df_merged = pd.merge(df_merged, df_meteo_location2, on='datetime', how='left')
df_merged = pd.merge(df_merged, df_meteo_location3_Jan,
                     on='datetime', how='left')
print(f"df_elec shape: {df_elec.shape}")
print(f"df_meteo shape: {df_meteo.shape}")
print(f"df_merged shape: {df_merged.shape}")
print(f"\nColumns in merged dataframe: {df_merged.shape[1]}")
print(f"Electricity columns: {df_elec.shape[1]}")
# -1 because period_end is not needed
print(f"Weather columns: {df_meteo.shape[1] - 1}")

df_elec shape: (52608, 237)
df_meteo shape: (52656, 8)
df_merged shape: (52608, 258)

Columns in merged dataframe: 258
Electricity columns: 237
Weather columns: 7


In [18]:
df_merged.columns.tolist()

['datetime',
 'HoD',
 'dow',
 'doy',
 'month',
 'year',
 '4_pv',
 '4_con',
 '4_net',
 '6_pv',
 '6_con',
 '6_net',
 '11_pv',
 '11_con',
 '11_net',
 '12_pv',
 '12_con',
 '12_net',
 '16_pv',
 '16_con',
 '16_net',
 '17_pv',
 '17_con',
 '17_net',
 '31_pv',
 '31_con',
 '31_net',
 '32_pv',
 '32_con',
 '32_net',
 '36_pv',
 '36_con',
 '36_net',
 '40_pv',
 '40_con',
 '40_net',
 '43_pv',
 '43_con',
 '43_net',
 '44_pv',
 '44_con',
 '44_net',
 '45_pv',
 '45_con',
 '45_net',
 '48_pv',
 '48_con',
 '48_net',
 '53_pv',
 '53_con',
 '53_net',
 '55_pv',
 '55_con',
 '55_net',
 '56_pv',
 '56_con',
 '56_net',
 '60_pv',
 '60_con',
 '60_net',
 '62_pv',
 '62_con',
 '62_net',
 '66_pv',
 '66_con',
 '66_net',
 '71_pv',
 '71_con',
 '71_net',
 '78_pv',
 '78_con',
 '78_net',
 '79_pv',
 '79_con',
 '79_net',
 '80_pv',
 '80_con',
 '80_net',
 '92_pv',
 '92_con',
 '92_net',
 '96_pv',
 '96_con',
 '96_net',
 '98_pv',
 '98_con',
 '98_net',
 '99_pv',
 '99_con',
 '99_net',
 '105_pv',
 '105_con',
 '105_net',
 '113_pv',
 '113_co

## Save the merged dataframe

In [19]:
df_merged.to_csv(
    '../data/ausgrid/ausgrid_merged_unprocessed.csv', index=False)